In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from plotly.subplots import make_subplots
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

In [ ]:
import os
os.environ["TABPFN_TOKEN"] = "************************************"

#Opening daily integrated data (QC > 0.75)

In [ ]:
Data = pd.read_csv('/content/drive/MyDrive/NEE_QC_75.csv', header = 0, decimal = '.', sep = ",", na_values = -9999)

In [ ]:
Data

,TIMESTAMP,TA,VPD,PPFD,LE,H,SW,NETRAD,NEE,SWVL1,...,STRD,LAI,Evavt,SWVL2,SWVL4,STL2,STL4,STL3,SWVL3,Rain
0,2002-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.450298,...,1.936148e+07,3.063915,-0.000039,0.414453,0.398721,301.078918,300.823182,301.273344,0.310388,0.375
1,2002-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.456160,...,1.906993e+07,3.061453,-0.000049,0.427195,0.398497,300.722744,300.830104,301.214389,0.312128,0.500
2,2002-01-03,23.763,0.708,84.049042,42.6066,12.8354,43.705,NaN,5.89980,0.466495,...,1.938431e+07,3.058990,-0.000041,0.461902,0.398326,300.078354,300.835657,301.111107,0.326004,45.500
3,2002-01-04,25.098,3.138,260.241130,46.6640,15.7407,131.087,NaN,2.16893,0.472386,...,1.902716e+07,3.056528,-0.000028,0.467553,0.398255,299.270851,300.838665,300.918914,0.345240,28.250
4,2002-01-05,25.076,2.597,314.081167,46.0852,25.6475,163.322,NaN,2.14902,0.471749,...,1.916936e+07,3.054065,-0.000022,0.470658,0.398177,299.264084,300.838253,300.712590,0.361726,39.750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3282,2011-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.403821,...,1.913217e+07,3.076155,-0.000050,0.368202,0.397184,301.109515,300.772685,301.224226,0.388818,1.625
3283,2011-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.432205,...,1.921129e+07,3.073693,-0.000031,0.374266,0.397059,300.857372,300.780044,301.178616,0.387808,3.750
3284,2011-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.420969,...,1.877759e+07,3.071230,-0.000054,0.385235,0.396936,300.704329,300.785977,301.107428,0.386909,0.000
3285,2011-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.423796,...,1.900162e+07,3.068768,-0.000051,0.386222,0.396759,300.832429,300.791402,301.060034,0.385631,0.000


In [ ]:
Data.columns

Index(['TIMESTAMP', 'TA', 'VPD', 'PPFD', 'LE', 'H', 'SW', 'NETRAD', 'NEE',
       'SWVL1', 'STL1', 'SSRD', 'STRD', 'LAI', 'Evavt', 'SWVL2', 'SWVL4',
       'STL2', 'STL4', 'STL3', 'SWVL3', 'Rain'],
      dtype='object')

In [ ]:
Data.columns = ["TIMESTAMP", "TA","VPD","PPFD","LE", "H","SW","NETRAD","NEE","SWVL1","STL1","SSRD","STRD","LAI","Evavt","SWVL2","SWVL4","STL2","STL4","STL3","SWVL3","Rain"]

In [ ]:
Data = Data.dropna()

In [ ]:
Data


,TIMESTAMP,TA,VPD,PPFD,LE,H,SW,NETRAD,NEE,SWVL1,...,STRD,LAI,Evavt,SWVL2,SWVL4,STL2,STL4,STL3,SWVL3,Rain
62,2002-03-04,23.490,2.603,271.349652,71.5430,16.91410,136.769,32.202261,3.500630,0.476435,...,1.907148e+07,2.602498,-0.000031,0.471117,0.482942,298.815773,300.122346,299.749830,0.486430,3.000
63,2002-03-05,24.050,4.157,405.416043,78.5416,22.69170,204.557,134.660875,1.378630,0.453615,...,1.881867e+07,2.605171,-0.000032,0.463903,0.485557,298.982045,300.114272,299.645934,0.488312,1.750
64,2002-03-06,23.694,3.597,340.538000,59.9588,12.75400,172.587,91.861217,1.433340,0.467868,...,1.875220e+07,2.607712,-0.000041,0.464852,0.486681,299.309706,300.105552,299.623412,0.482236,23.375
65,2002-03-07,24.245,3.439,346.292217,62.8135,15.07100,177.724,105.747750,2.312480,0.463534,...,1.883657e+07,2.610317,-0.000041,0.471624,0.487458,298.817235,300.095853,299.562910,0.481462,5.625
66,2002-03-08,25.472,5.047,424.357870,80.3522,18.52060,216.783,125.656913,2.163020,0.461699,...,1.873700e+07,2.612924,-0.000047,0.474315,0.487525,299.305305,300.085205,299.522301,0.476801,3.750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3211,2011-10-17,25.647,5.506,356.766895,54.0187,9.25173,209.070,88.515261,-2.715250,0.396030,...,1.913871e+07,3.920163,-0.000051,0.294924,0.406133,302.666122,300.669742,301.929460,0.296895,5.000
3212,2011-10-18,25.032,5.403,484.511870,90.4958,40.69350,241.955,150.697750,-2.826270,0.440301,...,1.912900e+07,3.908358,-0.000067,0.323315,0.405842,301.224890,300.689704,301.878686,0.296730,33.250
3215,2011-10-21,24.158,2.595,182.547167,47.8042,11.10300,85.306,68.058067,0.391046,0.449522,...,1.892360e+07,3.873008,-0.000034,0.380633,0.405115,300.063269,300.734176,301.367642,0.297451,0.000
3216,2011-10-22,24.044,2.952,193.560174,41.6949,2.96479,96.731,40.343250,0.058786,0.456838,...,1.892890e+07,3.861139,-0.000042,0.448842,0.404950,299.293798,300.742923,301.154134,0.304101,44.500


In [ ]:
Fold_1 = Data[(Data['TIMESTAMP'] > '2002-01-01') & (Data['TIMESTAMP'] < '2002-06-01')]
Fold_2 = Data[(Data['TIMESTAMP'] > '2002-05-31') & (Data['TIMESTAMP'] < '2002-11-01')]
Fold_3 = Data[(Data['TIMESTAMP'] > '2002-10-31') & (Data['TIMESTAMP'] < '2003-06-01')]
Fold_4 = Data[(Data['TIMESTAMP'] > '2003-05-31') & (Data['TIMESTAMP'] < '2003-11-01')]
Fold_5 = Data[(Data['TIMESTAMP'] > '2003-10-31') & (Data['TIMESTAMP'] < '2004-06-01')]
Fold_6 = Data[(Data['TIMESTAMP'] > '2004-05-31') & (Data['TIMESTAMP'] < '2004-11-01')]
Fold_7 = Data[(Data['TIMESTAMP'] > '2004-10-31') & (Data['TIMESTAMP'] < '2005-06-01')]
Fold_8 = Data[(Data['TIMESTAMP'] > '2005-10-31') & (Data['TIMESTAMP'] < '2006-06-01')]
Fold_9 = Data[(Data['TIMESTAMP'] > '2008-10-31') & (Data['TIMESTAMP'] < '2009-06-01')]
Fold_10 = Data[(Data['TIMESTAMP'] > '2009-05-31') & (Data['TIMESTAMP'] < '2009-11-01')]

# --- NOVOS SPLITS SAZONAIS ---
Wet_Train = pd.concat([Fold_1, Fold_3, Fold_5])
Wet_Val = pd.concat([Fold_7, Fold_8, Fold_9])

Dry_Train = pd.concat([Fold_2, Fold_4, Fold_6])
Dry_Val = Fold_10


# Lista de tuplas contendo: (Nome_do_Cenario, Dados_de_Treino, Dados_de_Validacao)
seasonal_scenarios = [
    ('Estacao_Chuvosa', Wet_Train, Wet_Val),
    ('Estacao_Seca', Dry_Train, Dry_Val)
]

In [ ]:
Historical_Data = Data[Data['TIMESTAMP'] < '2010-01-01'].copy()

X = Historical_Data.drop(columns=['TIMESTAMP', 'NEE'])
y = Historical_Data['NEE']
features = X.columns.tolist()

In [ ]:
%pip install "tabpfn @ git+https://github.com/PriorLabs/TabPFN.git" --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
%pip install shap --quiet

In [ ]:
import shap

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Assuming there is a TabPFNRegressor (if not, a different regressor should be used)
from tabpfn import TabPFNRegressor

In [ ]:
# 1. Preparação dos DataFrames de Treino Final e Teste
Final_Train_df = pd.concat([Wet_Train, Dry_Train, Wet_Val, Dry_Val])
X_train_final = Final_Train_df.drop(columns=['TIMESTAMP', 'NEE'])
y_train_final = Final_Train_df['NEE']

Test_df = Data[Data['TIMESTAMP'] >= '2010-01-01'].copy()
X_test_final = Test_df.drop(columns=['TIMESTAMP', 'NEE'])
y_test_final = Test_df['NEE']

mean_X_final, std_X_final = X_train_final.mean(), X_train_final.std()
mean_Y_final, std_Y_final = y_train_final.mean(), y_train_final.std()

X_train_final_norm = (X_train_final - mean_X_final) / std_X_final
Y_train_final_norm = (y_train_final - mean_Y_final) / std_Y_final
X_test_final_norm = (X_test_final - mean_X_final) / std_X_final

# ATENÇÃO: O TabPFN padrão impõe limite de linhas (ex: max 10.000 linhas no treino).
# Se o seu treino for muito grande, faça uma amostragem estratificada ou aleatória:
if len(X_train_final_norm) > 10000:
    print("Reduzindo o treino para 10.000 linhas devido à restrição arquitetural do TabPFN...")
    sample_idx = np.random.choice(len(X_train_final_norm), 10000, replace=False)
    X_tr_pfn = X_train_final_norm.iloc[sample_idx]
    y_tr_pfn = Y_train_final_norm.iloc[sample_idx]
else:
    X_tr_pfn = X_train_final_norm
    y_tr_pfn = Y_train_final_norm

# Inicializa o regressor TabPFN (forçando o uso de GPU se disponível)
# Initialize the regressor
regressor = TabPFNRegressor()
regressor.fit(X_tr_pfn, y_tr_pfn)

print("Prevendo com TabPFN...")
y_pred_pfn_norm = regressor.predict(X_test_final_norm)

# Desnormalizar
y_pred_pfn = (y_pred_pfn_norm * std_Y_final) + mean_Y_final

df_test_final = pd.DataFrame({
    'Data': Test_df['TIMESTAMP'],
    'NEE_True': y_test_final,
    'NEE_TabPFN': y_pred_pfn
})

df_test_final.to_csv('/content/drive/MyDrive/tabpfn_qc75.csv', index=False)

# Avaliação
print(f"TabPFN R²:   {r2_score(y_test_final, y_pred_pfn):.4f}")
print(f"TabPFN RMSE: {root_mean_squared_error(y_test_final, y_pred_pfn):.4f}")
print(f"TabPFN MSE: {mean_squared_error(y_test_final, y_pred_pfn):.4f}")

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Prevendo com TabPFN...
TabPFN R²:   0.5456
TabPFN RMSE: 1.0659
TabPFN MSE: 1.1362


In [ ]:
%pip install tabpfn tabpfn_extensions shap matplotlib scikit-learn --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%pip install tabpfn-client "tabpfn-extensions[interpretability]" --quiet

In [ ]:
from tabpfn_extensions.interpretability.shapiq import get_tabpfn_imputation_explainer

In [ ]:
import matplotlib as mpl

In [ ]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning # ou avisos genéricos de user warning

# Silencia os avisos de validação do scikit-learn e do shapiq
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import warnings
from sklearn.exceptions import InconsistentVersionWarning # ou avisos genéricos de user warning

# Silencia os avisos de validação do scikit-learn e do shapiq
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Ajuste do modelo e do explainer
reg = TabPFNRegressor(fit_mode="fit_with_cache")
reg.fit(X_tr_pfn, y_tr_pfn)

print("Instanciando o imputation explainer do TabPFN...")
explainer = get_tabpfn_imputation_explainer(
    model=reg,
    data=X_tr_pfn,
    index="SV",
)

feature_names = X_tr_pfn.columns
mpl.rc('figure', max_open_warning=0)

# 2. Selecionar o conjunto a ser explicado
# (Se o treino for grande, ex: 1220 linhas, pegar uma amostra representativa de 100-200 linhas acelera drasticamente)
X_eval = X_tr_pfn # Ou X_tr_pfn.iloc[:150] se quiser testar mais rápido

print(f"Calculando os valores SHAP para {len(X_eval)} instâncias...")
shap_values_list = []

# Iterando linha por linha para extrair os valores SHAP com o shapiq
for i in range(len(X_eval)):
    x_single = X_eval.iloc[i:i+1]  # <--- Removido o .values aqui
    # Explica a instância atual
    explanation = explainer.explain(x_single, budget=128)
    # O shapiq retorna um objeto ou array contendo os valores de 1ª ordem (SV)
    shap_values_list.append(explanation.values if hasattr(explanation, "values") else explanation)

# Junta tudo em uma matriz 2D e descarta a coluna extra de viés (se houver)
sv_matrix = np.vstack(shap_values_list)
if sv_matrix.shape[1] > X_eval.shape[1]:
    sv_matrix = sv_matrix[:, :-1]

# 3. Geração dos Gráficos
print("Salvando gráficos SHAP do TabPFN...")

# --- Beeswarm Plot ---
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(sv_matrix, X_eval, feature_names=feature_names, show=False, max_display=10)

plt.xlabel(r"SHAP values (Contributions to NEE, gC $m^{-2}$ $d^{-1}$)", fontsize=14)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout()

fig.savefig('/content/drive/MyDrive/beeswarm_TabPFN_qc75.png', dpi=400, bbox_inches='tight')
plt.close(fig)

# --- Bar Plot ---
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(sv_matrix, X_eval, feature_names=feature_names, plot_type="bar", show=False, max_display=10)

plt.xlabel(r"Mean (|SHAP value|) (Mean contributions to NEE, gC $m^{-2}$ $d^{-1}$)", fontsize=14)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout()

fig.savefig('/content/drive/MyDrive/shap_TabPFN_qc75.png', dpi=400, bbox_inches='tight')
plt.close(fig)

print("Gráficos SHAP do TabPFN salvos com sucesso!")

Instanciando o imputation explainer do TabPFN...
Calculando os valores SHAP para 530 instâncias...
Salvando gráficos SHAP do TabPFN...
Gráficos SHAP do TabPFN salvos com sucesso!


In [ ]:
from tabpfn.finetuning import FinetunedTabPFNRegressor

In [ ]:
# ==============================================================================
# --- 1. AVALIAÇÃO POR ESTAÇÃO (seasonal_scenarios) ---
# ==============================================================================
print("=======================================================")
print("--- 1. DESEMPENHO POR ESTAÇÃO (Folds Sazonais) ---")
print("=======================================================")

fold_results = []
fold_mse, fold_rmse, fold_r2 = [], [], []

for season_name, train_data, val_data in seasonal_scenarios:
    if train_data.empty or val_data.empty:
        continue

    num_cols = [c for c in train_data.columns if c not in ['TIMESTAMP', 'NEE']]

    X_train = train_data[num_cols].values.astype(np.float32)
    y_train = train_data['NEE'].values.astype(np.float32)
    X_val = val_data[num_cols].values.astype(np.float32)
    y_val = val_data['NEE'].values.astype(np.float32)

    # Normalização estatística do fold
    mean_X, std_X = X_train.mean(axis=0), X_train.std(axis=0)
    std_X = np.where(std_X == 0, 1.0, std_X)
    mean_Y, std_Y = y_train.mean(), y_train.std()
    if std_Y == 0: std_Y = 1.0

    X_train_norm = (X_train - mean_X) / std_X
    Y_train_norm = (y_train - mean_Y) / std_Y
    X_val_norm = (X_val - mean_X) / std_X
    Y_val_norm = (y_val - mean_Y) / std_Y

    # Limite arquitetural do TabPFN por fold
    max_samples = 10000
    if len(X_train_norm) > max_samples:
        sample_idx = np.random.choice(len(X_train_norm), max_samples, replace=False)
        X_tr_fold = X_train_norm[sample_idx]
        y_tr_fold = Y_train_norm[sample_idx]
    else:
        X_tr_fold = X_train_norm
        y_tr_fold = Y_train_norm

    # Fine-Tuning do TabPFN (instância nova a cada fold/estação)
    finetuned_reg = FinetunedTabPFNRegressor(
        epochs=50,
        learning_rate=1e-5,
        validation_split_ratio=0.0,
        device="cuda",
        n_estimators_finetune = 8,
        n_estimators_validation = 16,
        n_estimators_final_inference = 16,
        early_stopping = True,
        early_stopping_patience = 8
    )

    finetuned_reg.fit(X_tr_fold, y_tr_fold)

    # Predição e desnormalização
    y_pred_val_norm = finetuned_reg.predict(X_val_norm)
    y_pred_val_real = (y_pred_val_norm * std_Y) + mean_Y

    # Métricas
    mse = mean_squared_error(y_val, y_pred_val_real)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, y_pred_val_real)

    fold_mse.append(mse)
    fold_rmse.append(rmse)
    fold_r2.append(r2)

    fold_results.append({
        'Estação/Cenário': season_name,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
    })

    print(f"Cenário: {season_name:<20} -> MSE: {mse:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")

df_folds_summary = pd.DataFrame(fold_results)
print("=======================================================")

# Resumo estatístico (Média e Desvio-Padrão amostral)
mean_mse, std_mse = np.mean(fold_mse), np.std(fold_mse, ddof=1) if len(fold_mse) > 1 else 0.0
mean_rmse, std_rmse = np.mean(fold_rmse), np.std(fold_rmse, ddof=1) if len(fold_rmse) > 1 else 0.0
mean_r2, std_r2 = np.mean(fold_r2), np.std(fold_r2, ddof=1) if len(fold_r2) > 1 else 0.0

print("\nMetrics Summary (Mean ± SD across Folds):")
print(f"MSE:  {mean_mse:.4f} ± {std_mse:.4f}")
print(f"RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}")
print(f"R2:   {mean_r2:.4f} ± {std_r2:.4f}")

print("\nMetrics across the folds:")
for i in range(len(fold_mse)):
    print(f"Fold {i+1:<2} ({fold_results[i]['Estação/Cenário']}) -> MSE: {fold_mse[i]:.4f} | RMSE: {fold_rmse[i]:.4f} | R2: {fold_r2[i]:.4f}")
print("=======================================================\n")

# ==============================================================================
# --- 2. TREINAMENTO FINAL E TESTE (2010+) ---
# ==============================================================================
print("=======================================================")
print("--- 2. AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (2010+) ---")
print("=======================================================")

Final_Train_df = pd.concat([Wet_Train, Dry_Train, Wet_Val, Dry_Val])
X_train_final = Final_Train_df.drop(columns=['TIMESTAMP', 'NEE'])
y_train_final = Final_Train_df['NEE']

Test_df = Data[Data['TIMESTAMP'] >= '2010-01-01'].copy()
X_test_final = Test_df.drop(columns=['TIMESTAMP', 'NEE'])
y_test_final = Test_df['NEE']

mean_X_final, std_X_final = X_train_final.mean(), X_train_final.std()
mean_Y_final, std_Y_final = y_train_final.mean(), y_train_final.std()

X_train_final_norm = (X_train_final - mean_X_final) / std_X_final
Y_train_final_norm = (y_train_final - mean_Y_final) / std_Y_final
X_test_final_norm = (X_test_final - mean_X_final) / std_X_final

if len(X_train_final_norm) > 10000:
    sample_idx = np.random.choice(len(X_train_final_norm), 10000, replace=False)
    X_tr_pfn = X_train_final_norm.iloc[sample_idx]
    y_tr_pfn = Y_train_final_norm.iloc[sample_idx]
else:
    X_tr_pfn = X_train_final_norm
    y_tr_pfn = Y_train_final_norm

# Modelo Final Ajustado
finetuned_reg_final = FinetunedTabPFNRegressor(
    epochs=50,
    learning_rate=1e-5,
    validation_split_ratio=0.0,
    device="cuda",
    n_estimators_finetune = 8,
    n_estimators_validation = 16,
    n_estimators_final_inference = 16,
)

finetuned_reg_final.fit(X_tr_pfn, y_tr_pfn)
y_pred_pfn_norm = finetuned_reg_final.predict(X_test_final_norm)
y_pred_pfn = (y_pred_pfn_norm * std_Y_final) + mean_Y_final

# Salvando o resultado para gráficos posteriores
df_test_final = pd.DataFrame({
    'Data': Test_df['TIMESTAMP'],
    'NEE_True': y_test_final,
    'NEE_TabPFN_Finetuned': y_pred_pfn
})
df_test_final.to_csv('/content/drive/MyDrive/tabpfn_finetuned_qc75.csv', index=False)

# Métricas Finais
final_mse = mean_squared_error(y_test_final, y_pred_pfn)
final_rmse = root_mean_squared_error(y_test_final, y_pred_pfn)
final_r2 = r2_score(y_test_final, y_pred_pfn)

print(f"TabPFN Finetuned R²:   {final_r2:.4f}")
print(f"TabPFN Finetuned RMSE: {final_rmse:.4f}")
print(f"TabPFN Finetuned MSE:  {final_mse:.4f}")
print("=======================================================")

--- 1. DESEMPENHO POR ESTAÇÃO (Folds Sazonais) ---


Finetuning Epoch 1/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 2/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 3/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 4/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 5/50:   0%|          | 0/1 [00:00<?, ?it/s]

... (144 outputs left out)

Finetuning Epoch 48/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 49/50:   0%|          | 0/1 [00:00<?, ?it/s]

Finetuning Epoch 50/50:   0%|          | 0/1 [00:00<?, ?it/s]

TabPFN Finetuned R²:   0.5593
TabPFN Finetuned RMSE: 1.0498
TabPFN Finetuned MSE:  1.1020


In [ ]:
print("Instanciando o imputation explainer do TabPFN...")
explainer = get_tabpfn_imputation_explainer(
    model=finetuned_reg_final,
    data=X_tr_pfn,
    index="SV",
)

feature_names = X_tr_pfn.columns
mpl.rc('figure', max_open_warning=0)

# 2. Selecionar o conjunto a ser explicado
# (Se o treino for grande, ex: 1220 linhas, pegar uma amostra representativa de 100-200 linhas acelera drasticamente)
X_eval = X_tr_pfn # Ou X_tr_pfn.iloc[:150] se quiser testar mais rápido

print(f"Calculando os valores SHAP para {len(X_eval)} instâncias...")
shap_values_list = []

# Iterando linha por linha para extrair os valores SHAP com o shapiq
for i in range(len(X_eval)):
    x_single = X_eval.iloc[i:i+1]  # <--- Removido o .values aqui
    # Explica a instância atual
    explanation = explainer.explain(x_single, budget=128)
    # O shapiq retorna um objeto ou array contendo os valores de 1ª ordem (SV)
    shap_values_list.append(explanation.values if hasattr(explanation, "values") else explanation)

# Junta tudo em uma matriz 2D e descarta a coluna extra de viés (se houver)
sv_matrix = np.vstack(shap_values_list)
if sv_matrix.shape[1] > X_eval.shape[1]:
    sv_matrix = sv_matrix[:, :-1]

# 3. Geração dos Gráficos
print("Salvando gráficos SHAP do TabPFN...")

# --- Beeswarm Plot ---
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(sv_matrix, X_eval, feature_names=feature_names, show=False, max_display=10)

plt.xlabel(r"SHAP values (Contributions to NEE, gC $m^{-2}$ $d^{-1}$)", fontsize=14)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout()

fig.savefig('/content/drive/MyDrive/beeswarm_TabPFN_tuned_qc75.png', dpi=400, bbox_inches='tight')
plt.close(fig)

# --- Bar Plot ---
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(sv_matrix, X_eval, feature_names=feature_names, plot_type="bar", show=False, max_display=10)

plt.xlabel(r"Mean (|SHAP value|) (Mean contributions to NEE, gC $m^{-2}$ $d^{-1}$)", fontsize=14)
plt.xticks(fontsize=12); plt.yticks(fontsize=12)
plt.tight_layout()

fig.savefig('/content/drive/MyDrive/shap_TabPFN_tuned_qc75.png', dpi=400, bbox_inches='tight')
plt.close(fig)

print("Gráficos SHAP do TabPFN salvos com sucesso!")

Instanciando o imputation explainer do TabPFN...
Calculando os valores SHAP para 530 instâncias...
Salvando gráficos SHAP do TabPFN...
Gráficos SHAP do TabPFN salvos com sucesso!
